In [1]:
import pandas as pd
import numpy as np
import random

# -------------------------- 请修改以下3个参数 --------------------------
raw_csv_path = "./语料库/wmt_zh_en_training_corpus.csv"  # 原始GB级CSV路径
output_10w_path = "./语料库/data_10w.csv"  # 输出路径（和训练路径一致）
chunk_size = 500000  # 分块大小（内存越小设越小，如8GB内存设2万-5万）
# -----------------------------------------------------------------------

def reservoir_sampling_large_csv(raw_path, output_path, sample_size=500000, encoding='utf-8'):
    """
    蓄水池采样：分块读取时直接随机采样，不加载全量数据（内存友好）
    """
    try:
        # 初始化“蓄水池”：存储最终要保留的10万条数据
        reservoir = []
        total_rows = 0  # 统计原始文件总行数（仅用于日志）
        
        print(f"正在分块采样原始CSV：{raw_path}")
        for i, chunk in enumerate(pd.read_csv(
            raw_path,
            chunksize=chunk_size,
            encoding=encoding,
            low_memory=False
        )):
            # 过滤空行（避免无效数据）
            chunk = chunk.dropna(how='all')  # 全空行删除
            chunk = chunk[chunk.iloc[:, 0].notna() & chunk.iloc[:, 1].notna()]  # 中英文列非空
            chunk_rows = len(chunk)
            total_rows += chunk_rows
            
            # 蓄水池采样逻辑
            if len(reservoir) < sample_size:
                # 蓄水池未满：直接加入当前块所有数据
                reservoir.append(chunk)
                current_reservoir_size = sum(len(c) for c in reservoir)
                # 若加入后超过10万条，截断到10万条
                if current_reservoir_size > sample_size:
                    combined = pd.concat(reservoir, ignore_index=True)
                    reservoir = [combined.sample(sample_size, random_state=42, ignore_index=True)]
            else:
                # 蓄水池已满：按概率替换现有数据（保证全局随机性）
                for _, row in chunk.iterrows():
                    total_rows_so_far = total_rows - chunk_rows + _ + 1
                    # 生成随机数：1~total_rows_so_far，若小于sample_size则替换
                    if random.randint(1, total_rows_so_far) <= sample_size:
                        replace_idx = random.randint(0, sample_size - 1)
                        reservoir[0].iloc[replace_idx] = row
            
            print(f"已处理第{i+1}块 | 累计读取行数：{total_rows:,} | 蓄水池当前大小：{len(reservoir[0]) if reservoir else 0:,}")
        
        # 最终合并蓄水池数据（确保是10万条）
        final_sample = pd.concat(reservoir, ignore_index=True).head(sample_size)
        print(f"\n最终采样数据量：{len(final_sample):,}条")
        
        # 保存（不保留索引）
        final_sample.to_csv(
            output_path,
            index=False,
            encoding=encoding
        )
        
        print(f"✅ 完成！新文件已保存到：{output_path}")
        print("数据前5行预览：")
        print(final_sample.head())
        
    except FileNotFoundError:
        print(f"❌ 错误：找不到原始文件，请检查路径：{raw_path}")
    except UnicodeDecodeError:
        print(f"❌ 错误：编码失败，尝试将 encoding 改为 'gbk'")
    except Exception as e:
        print(f"❌ 其他错误：{str(e)}")

# 执行采样（随机种子已固定，结果可复现）
random.seed(42)
np.random.seed(42)
reservoir_sampling_large_csv(raw_csv_path, output_10w_path)

正在分块采样原始CSV：./语料库/wmt_zh_en_training_corpus.csv
已处理第1块 | 累计读取行数：500,000 | 蓄水池当前大小：500,000
已处理第2块 | 累计读取行数：1,000,000 | 蓄水池当前大小：500,000
已处理第3块 | 累计读取行数：1,500,000 | 蓄水池当前大小：500,000
已处理第4块 | 累计读取行数：2,000,000 | 蓄水池当前大小：500,000
已处理第5块 | 累计读取行数：2,500,000 | 蓄水池当前大小：500,000
已处理第6块 | 累计读取行数：3,000,000 | 蓄水池当前大小：500,000
已处理第7块 | 累计读取行数：3,500,000 | 蓄水池当前大小：500,000
已处理第8块 | 累计读取行数：4,000,000 | 蓄水池当前大小：500,000
已处理第9块 | 累计读取行数：4,500,000 | 蓄水池当前大小：500,000
已处理第10块 | 累计读取行数：5,000,000 | 蓄水池当前大小：500,000
已处理第11块 | 累计读取行数：5,500,000 | 蓄水池当前大小：500,000
已处理第12块 | 累计读取行数：6,000,000 | 蓄水池当前大小：500,000
已处理第13块 | 累计读取行数：6,500,000 | 蓄水池当前大小：500,000
已处理第14块 | 累计读取行数：7,000,000 | 蓄水池当前大小：500,000
已处理第15块 | 累计读取行数：7,500,000 | 蓄水池当前大小：500,000
已处理第16块 | 累计读取行数：8,000,000 | 蓄水池当前大小：500,000
已处理第17块 | 累计读取行数：8,500,000 | 蓄水池当前大小：500,000
已处理第18块 | 累计读取行数：9,000,000 | 蓄水池当前大小：500,000
已处理第19块 | 累计读取行数：9,500,000 | 蓄水池当前大小：500,000
已处理第20块 | 累计读取行数：9,999,999 | 蓄水池当前大小：500,000
已处理第21块 | 累计读取行数：10,499,998 | 蓄水池当前大小：500,000
已处理第22块 | 累计读取行数：

In [2]:
import pandas as pd

data=pd.read_csv("./语料库/data_10w.csv")
data.head

<bound method NDFrame.head of                                                         0  \
0                                  ㈦ 制定 商品 融资 和 风险管理 办法 ；   
1                         D . 安全 理事会 与 国际 刑事警察 组织 之间 的 合作   
2       109 . 造成 本项 下 差异 的 主要 因素 是 2007 / 08 年度 和 2008...   
3       纪念 世界 土壤 日 ( 12 月 5 日 ) 和 启动 国际 土壤 年 特别 活动 ( 大...   
4       匈牙利 对 《 语言 法 》 的 限制性 规定 表示 关切 ， 敦促 斯洛伐克 通过 更加 ...   
...                                                   ...   
499995  13 . 将 继续 根据 联合国 国家 工作队 的 需求 ， 由 指导 委员会 和 协调 小...   
499996                              A / 58 / PV . 56 和 92   
499997  资金 转移 可以 是 现金 方式 ， 也 可 通过 贸易 公司 或 空壳 公司 在 地方 、...   
499998  三是 加大 依法 支付 农民工 工资 的 宣传 力度 ， 大力 推动 用人单位 全面落实 "...   
499999  还 回顾 《 联合国 打击 跨国 有 组织 犯罪 公约 》 及 《 联合国 打击 跨国 有 ...   

                                                        1  
0       ( VII ) Develop commodity financing and risk m...  
1       D. Cooperation between the Security Council an...  
2       109 . the main factor contributing to the vari...